# XGBoost

We're going back with our clean dataset (without binaries). This time, we are going to sum the total estimated owners per genre/category/tag. This is a different approach.

In [ ]:
import pandas as pd
import numpy as np
import ast
import joblib
import json
import os
from collections import defaultdict
from itertools import combinations
from xgboost import XGBRegressor

c:\Users\monta\Documents\Projets_code\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Importing the clean dataset
df = pd.read_csv("dataset/clean_steam_dataset.csv")

In [ ]:
#Reused the convert function from dataset_cleaning to help read properly the values from genres, categories & tags
def convert(value):
    """Convert list/dictionary string to Python object"""
    try:
        return ast.literal_eval(value) if isinstance(value, str) else value
    except Exception as e:
        #empty list for failed conversions
        print(f"Conversion failed. Error : {e}")
        return []

In [3]:
os.makedirs("results", exist_ok=True)

### GENRES

In [8]:
df["genres"] = df["genres"].apply(convert)

In [9]:
#checking if the genres are lists, and not str

print(df['genres'].head(10))
print(df['genres'].apply(type).value_counts())

0    [Action, Adventure, Massively Multiplayer, Fre...
1                                  [Action, Adventure]
2                                             [Action]
3    [Action, Adventure, Indie, Massively Multiplay...
4                    [Action, Adventure, Free To Play]
5                             [Action, Adventure, RPG]
6                                                [RPG]
7                                        [Action, RPG]
8                             [Indie, RPG, Simulation]
9                                                [RPG]
Name: genres, dtype: object
genres
<class 'list'>    86847
Name: count, dtype: int64


In [ ]:
#getting the data from the main columns of interest (genres, owners & years), ignoring the values that may add noise
genre_popularity = defaultdict(lambda: defaultdict(float))

for _, row in df.iterrows():
    year = row.get("release_year")
    owners = row.get("total_est_owners", 0)
    genres = row.get("genres", [])
    
    if not isinstance(genres, list):
        continue
    if pd.isna(year) or year == 0:
        continue

    for g in genres:
        genre_popularity[g][year] += owners

In [11]:
print(len(genre_popularity))
print(list(genre_popularity.keys())[:10])

33
['Action', 'Adventure', 'Massively Multiplayer', 'Free To Play', 'Indie', 'RPG', 'Simulation', 'Strategy', 'Early Access', 'Casual']


In [ ]:
for g, data in genre_popularity.items():
    if len(data) > 2:
        print(g, list(data.items())[:5])
        break

Action [(2017, 574780000.0), (2015, 411020000.0), (2018, 365825000.0), (2020, 478100000.0), (2024, 532145000.0)]


In [ ]:
target_year = 2035 #year we want to study
genre_forecasts = {} #the predictions

for genre, year_scores in genre_popularity.items():
    years = np.array(list(year_scores.keys()))
    scores = np.array(list(year_scores.values()))

    #skipping genres appearing in too few years
    if len(years) < 3:
        continue

    #preparing for xgboost
    X = years.reshape(-1, 1)
    y = scores

    genre_model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    genre_model.fit(X, y)

    predictions = genre_model.predict(np.array([[target_year]]))[0]

    genre_forecasts[genre] = predictions

In [ ]:
#calculating the probabilities for the X future genres to be popular/appear in games
total = sum(genre_forecasts.values())
genre_percentages = {g: (v / total) * 100 for g, v in genre_forecasts.items()}

sorted_pct = sorted(genre_percentages.items(), key=lambda x: x[1], reverse=True)

#creating variable that we will save in JSON format a bit later
top_predictions = [
    {"genre": genre, "predicted_percentage":getattr(pct, "tolist", lambda: pct)()} #getattr because JSON doesn't accept numpy format
    for genre, pct in sorted_pct[:10]
]

print("Predicted genre popularity in 2035 (as % of total):")
for genre, pct in sorted_pct[:10]: #only getting top 10
    print(f"{genre:20s} — {pct:.2f}%")


Predicted genre popularity in 2035 (as % of total):
Action               — 20.46%
Indie                — 16.66%
RPG                  — 16.02%
Adventure            — 12.23%
Casual               — 9.39%
Simulation           — 8.32%
Strategy             — 5.16%
Early Access         — 2.71%
Racing               — 2.71%
Massively Multiplayer — 2.43%


In [ ]:
#Saving the model

joblib.dump(genre_model, "results/xgb_genre_model.joblib")
print(f"Saved model to results/xgb_genre_model.joblib")

Saved model to results/xgb_genre_model.joblib


In [ ]:
#Getting the results with the target year, and the predictions we saw just above
results = {
    "target_year":target_year,
    "top_predictions":top_predictions
}

In [ ]:
#Saving the results in JSON format

with open(f"results/xgb_genre_model_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)
print(f"Saved model to results/xgb_genre_model_results.joblib")

Saved model to results/xgb_genre_model_results.joblib


### CATEGORIES

The process is the same as for "Genres"

In [26]:
df["categories"] = df["categories"].apply(convert)

In [27]:
#checking if the genres are lists, and not str

print(df['categories'].head(10))
print(df['categories'].apply(type).value_counts())

0    [Multi-player, PvP, Online PvP, Stats, Remote ...
1    [Single-player, Multi-player, PvP, Online PvP,...
2    [Single-player, Multi-player, PvP, Online PvP,...
3    [Multi-player, MMO, PvP, Online PvP, Co-op, On...
4    [Multi-player, PvP, Online PvP, Co-op, Online ...
5    [Single-player, Steam Achievements, Full contr...
6    [Single-player, Steam Achievements, Steam Trad...
7    [Single-player, Multi-player, PvP, Online PvP,...
8    [Single-player, Multi-player, Co-op, Online Co...
9    [Single-player, Steam Achievements, Full contr...
Name: categories, dtype: object
categories
<class 'list'>    86847
Name: count, dtype: int64


In [28]:
cat_popularity = defaultdict(lambda: defaultdict(float))

for _, row in df.iterrows():
    year = row.get("release_year")
    owners = row.get("total_est_owners", 0)
    categories = row.get("categories", [])
    
    if not isinstance(categories, list):
        continue
    if pd.isna(year) or year == 0:
        continue

    for c in categories:
        cat_popularity[c][year] += owners

In [29]:
print(len(cat_popularity))
print(list(cat_popularity.keys())[:10])

40
['Multi-player', 'PvP', 'Online PvP', 'Stats', 'Remote Play on Phone', 'Remote Play on Tablet', 'Single-player', 'Co-op', 'Online Co-op', 'Steam Achievements']


In [30]:
for c, data in cat_popularity.items():
    if len(data) > 2:
        print(c, list(data.items())[:5])
        break

Multi-player [(2017, 532530000.0), (2015, 362315000.0), (2018, 363955000.0), (2020, 409420000.0), (2022, 365615000.0)]


In [31]:
target_year = 2035
cat_forecasts = {}

for category, year_scores in cat_popularity.items():
    years = np.array(list(year_scores.keys()))
    scores = np.array(list(year_scores.values()))

    #skipping genres appearing in too few years
    if len(years) < 3:
        continue

    X = years.reshape(-1, 1)
    y = scores

    cat_model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    cat_model.fit(X, y)

    predictions = cat_model.predict(np.array([[target_year]]))[0]

    cat_forecasts[category] = predictions

In [32]:
total = sum(cat_forecasts.values())
cat_pct = {c: (v / total) * 100 for c, v in cat_forecasts.items()}

sorted_pct = sorted(cat_pct.items(), key=lambda x: x[1], reverse=True)

top_predictions = [
    {"category": category, "predicted_percentage":getattr(pct, "tolist", lambda: pct)()}
    for category, pct in sorted_pct[:10]
]

print("Predicted category popularity in 2035 (as % of total):")
for category, pct in sorted_pct[:10]:
    print(f"{category:20s} — {pct:.2f}%")

Predicted category popularity in 2035 (as % of total):
Family Sharing       — 13.19%
Single-player        — 11.98%
Steam Achievements   — 9.02%
Steam Cloud          — 7.10%
Multi-player         — 5.22%
Co-op                — 5.16%
Full controller support — 5.14%
Steam Trading Cards  — 4.66%
Online Co-op         — 4.59%
PvP                  — 4.48%


In [33]:
joblib.dump(cat_model, "results/xgb_cat_model.joblib")
print(f"Saved model to results/xgb_cat_model.joblib")

Saved model to results/xgb_cat_model.joblib


In [34]:
results = {
    "target_year":target_year,
    "top_predictions":top_predictions
}

In [35]:
with open(f"results/xgb_cat_model_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)
print(f"Saved model to results/xgb_cat_model_results.joblib")

Saved model to results/xgb_cat_model_results.joblib


### TAG COMBINATIONS (triplets)

This is a different case, as we are now trying to get tag triplets for a more precise idea of the future trending tags/games. We will first try training xgboost and see the kinds of results we get.

In [36]:
df["tags"] = df["tags"].apply(convert)

In [37]:
tag_popularity = defaultdict(lambda: defaultdict(float))

for _, row in df.iterrows():
    year = row.get("release_year")
    owners = row.get("total_est_owners", 0)
    tags = row.get("tags", {})

    if not isinstance(tags, dict):
        continue
    if pd.isna(year) or year == 0:
        continue

    # getting the top 5 tags per game
    top_tags = sorted(tags.items(), key=lambda x: x[1], reverse=True)[:5]
    top_tag_names = [t[0] for t in top_tags]

    # creating all tag triplets
    for combo in combinations(top_tag_names, 3):
        tag_popularity[combo][year] += owners

In [38]:
print(len(tag_popularity))
print(list(tag_popularity.keys())[:10])

263638
[('Survival', 'Shooter', 'Battle Royale'), ('Survival', 'Shooter', 'Multiplayer'), ('Survival', 'Shooter', 'FPS'), ('Survival', 'Battle Royale', 'Multiplayer'), ('Survival', 'Battle Royale', 'FPS'), ('Survival', 'Multiplayer', 'FPS'), ('Shooter', 'Battle Royale', 'Multiplayer'), ('Shooter', 'Battle Royale', 'FPS'), ('Shooter', 'Multiplayer', 'FPS'), ('Battle Royale', 'Multiplayer', 'FPS')]


In [39]:
target_year = 2035
predictions = []

for combo, year_scores in tag_popularity.items():
    years = np.array(list(year_scores.keys()))
    scores = np.array(list(year_scores.values()))

    if len(years) < 3:
        continue

    X = years.reshape(-1, 1)
    y = scores

    tag_model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )

    tag_model.fit(X, y)

    pred = tag_model.predict(np.array([[target_year]]))[0]

    predictions.append((combo, pred))

In [43]:
total = sum(score for _, score in predictions)
triplet_percentages = {combo: (score / total) * 100 for combo, score in predictions}
sorted_pct = sorted(triplet_percentages.items(), key=lambda x: x[1], reverse=True)

print(f"Top predicted tag combinations in 2035 (in % from all existing tag combinations):")
for i, (combo, pct) in enumerate(sorted_pct[:10], 1):
    combo_str = ", ".join(combo)
    print(f"{i}. {combo_str:60s} — {pct:.2f}%")

top_predictions = [
    {"tag_triplet": combo, "predicted_percentage":getattr(pct, "tolist", lambda: pct)()}
    for combo, pct in sorted_pct[:10]
]

Top predicted tag combinations in 2035 (in % from all existing tag combinations):
1. Survival, Multiplayer, Open World Survival Craft             — 3.01%
2. Open World, Survival, Multiplayer                            — 2.63%
3. Battle Royale, Multiplayer, FPS                              — 2.51%
4. Open World, Multiplayer, Open World Survival Craft           — 2.51%
5. MMORPG, RPG, Adventure                                       — 2.51%
6. Open World, Survival, Open World Survival Craft              — 2.51%
7. Open World, Massively Multiplayer, RPG                       — 2.51%
8. Action RPG, Action, RPG                                      — 2.51%
9. Action RPG, Hack and Slash, RPG                              — 1.42%
10. Action RPG, RPG, Multiplayer                                 — 1.17%


In [40]:
joblib.dump(tag_model, "results/xgb_tag_model.joblib")
print(f"Saved model to results/xgb_tag_model.joblib")

Saved model to results/xgb_tag_model.joblib


In [44]:
results = {
    "target_year":target_year,
    "top_predictions":top_predictions
}

In [45]:
with open(f"results/xgb_tag_model_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)
print(f"Saved model to results/xgb_tag_model_results.joblib")

Saved model to results/xgb_tag_model_results.joblib


We can observe a couple of things from the tag triplets results. 

**1.** Some tag triplets are very alike. 

-> "1. Survival, Multiplayer, Open World Survival Craft", "4. Open World, Multiplayer, Open World Survival Craft", "6. Open World, Survival, Open World Survival Craft"...

**2.** From 3 to 8, all of the tags have the same percentage

**3.** The percentages could be improved to make them more understandable

**4.** The results have to be taken with a grain of salt

For later tweaks, we could implement a **NLP algorithm** that could encode the tags, and only predict tags of main categories if their values are too close. 

For example, "Open World" and "Open World Survival Craft" are very similar (open world), but they aren't totally the same. However, there is no need for them to be in the same tag triplets. Thus, we could simply remove the "Open World" aspect. For number 6, we could simply keep "Open World Survival Craft", and replace the other two with other tags that could define the game a bit better.

Encoding tags and getting their embeddings could be a great way to get better results.

Again, those results are not 100% accurate. We have to remember that XGBoost simply tried to find patterns, and was based on the total estimated owners column, which means it didn't take into account game popularity peaks, trends, sales... Moreoever, there is a strong randomness factor.

---------

For later works, we can try the following:

- implement an NLP algorithm
- improve percentages
- study game popularity peaks
- augment the dataset with the remaining 2025 games
- try other models (RandomForest, basic scikit-learn, LLMs...)